# 08 - 고전-양자 하이브리드 모델: 앞단 고전 레이어까지의 역전파

앞단의 고전 `torch.nn.Linear` 레이어가 컴파일된 PADO-Pauli 프로그램의 임베딩 각도를 생성하고, 단 한 번의 `backward()` 호출로
고전 레이어와 양자 회로를 동시에 학습합니다. 컴파일된 프로그램이 일반 PyTorch 텐서 연산으로 동작하기 때문에,
`loss.backward()`를 호출하면 회로 파라미터뿐만 아니라 앞단의 고전 가중치까지 gradient가 그대로 전달됩니다.

```
x (특징 2개) -> nn.Linear(2, 4) -> 임베딩 각도 -> 컴파일된 프로그램 (thetas) -> <Z0 Z1> -> loss
```

이 노트북에서는 다음 세 가지를 확인합니다.
1. 고전 가중치로 전달된 gradient가 정확한지 검증(중앙차분법과 비교),
2. 공동 학습 시 두 파라미터 그룹이 모두 업데이트되며 손실(loss)이 감소하는지 확인,
3. `embedding.detach()`를 사용했을 때 설명대로 앞단 고전 레이어의 가중치가 고정되는지 확인.

- 회로는 고전 레이어의 출력값 4개를 두 큐비트의 `RY` 회전 게이트에 인코딩합니다(`CNOT` 전후로 2개씩 입력). 이어서
  학습 가능한 `RY` 회전 게이트 2개를 적용한 뒤 $Z_0Z_1$ 기댓값을 측정합니다.

In [1]:
# 초기 설정: GPU 전용 실행, float64 정밀도, 재현성을 위한 시드 고정


import warnings
warnings.filterwarnings('ignore')

import torch
from padopauli import Circuit

if not torch.cuda.is_available():
    raise RuntimeError('This notebook is configured for GPU-only execution, but CUDA is not available.')

torch.manual_seed(0)
device = 'cuda'
dtype = torch.float64
print(torch.cuda.get_device_name(0))


AMD Instinct MI300X VF


In [2]:
# 회로는 한 번만 컴파일합니다. embedding_idx=k는 고전 레이어 출력의 k번째 원소를, param_idx=i는 thetas[i]를 참조합니다.


qc = Circuit(n_qubits=2)
qc.ry(0, embedding_idx=0)
qc.ry(1, embedding_idx=1)
qc.cnot(0, 1)
qc.ry(0, embedding_idx=2)   # 데이터 재업로드(data re-uploading): 고전 특징을 다시 인코딩
qc.ry(1, embedding_idx=3)
qc.ry(0, param_idx=0)
qc.ry(1, param_idx=1)
qc.compile(observables=[('ZZ', [0, 1])], preset='gpu')


propagate:   0%|          | 0/7 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 3


zero-filter:   0%|          | 0/7 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 3 (100.000000% of peak)


CompiledProgram(circuit=[RY[0](prior[0]), RY[1](prior[1]), CNOT[0, 1], RY[0](prior[2]), RY[1](prior[3]), RY[0](θ[0]), RY[1](θ[1])], psum_union=TensorOperatorProgram(n_qubits=2, x_mask=tensor([0, 0, 0], device='cuda:0'), coeff_init=tensor([1.], device='cuda:0', dtype=torch.float64), steps=[CompactStep(mat_const=SignedPerm(shape=(2, 1), row=tensor([], device='cuda:0', dtype=torch.int32), col=tensor([], device='cuda:0', dtype=torch.int32), val=tensor([], device='cuda:0', dtype=torch.int8), rows_unique=True, cols_unique=True), mat_sin=SignedPerm(shape=(2, 1), row=tensor([1], device='cuda:0', dtype=torch.int32), col=None, val=tensor([-1], device='cuda:0', dtype=torch.int8), rows_unique=True, cols_unique=True), param_idx=1, shape=(2, 1), embedding_idx=-1, unchanged_cols=tensor([0], device='cuda:0', dtype=torch.int32), n_comm=0, unchanged_unique=True), CompactStep(mat_const=SignedPerm(shape=(4, 2), row=tensor([], device='cuda:0', dtype=torch.int32), col=tensor([], device='cuda:0', dtype=torch

In [3]:
# 하이브리드 모델 구성: lin은 앞단의 고전 레이어이고, thetas는 학습할 회로 회전 각도입니다.


lin = torch.nn.Linear(2, 4).to(device=device, dtype=dtype)
thetas = torch.nn.Parameter(torch.zeros(2, dtype=dtype, device=device))

X = (torch.rand(256, 2, dtype=dtype, device=device) * 2 - 1)
y = ((X[:, 0] * X[:, 1]) > 0).to(dtype)          # XOR 형태의 타깃 라벨

def loss_fn():
    angles = lin(X)                               # 고전 레이어 출력 -> 임베딩 각도
    expv = qc.expvals(thetas, embedding=angles)[:, 0]
    prob = torch.clamp((expv + 1) * 0.5, 1e-6, 1 - 1e-6)
    return torch.nn.functional.binary_cross_entropy(prob, y)

print('loss at initialization:', float(loss_fn()))


loss at initialization: 1.7090882410369925


## 1) 고전 가중치로 전달되는 gradient 검증

한 번의 backward 호출로 계산된 `lin.weight`의 모든 gradient 원소를 동일한 손실 함수에 대한 중앙차분(central difference) 결과와 비교합니다. 두 값이 정밀하게 일치한다는 것은
컴파일된 양자 프로그램을 거쳐 앞단 고전 레이어까지 연쇄 법칙(chain rule)이 정확히 작동함을 의미합니다.

In [4]:
# 단일 backward 호출로 구한 해석적 gradient를 중앙차분 근사값과 비교합니다.


for t in (lin.weight, lin.bias, thetas):
    t.grad = None
loss_fn().backward()
g_analytic = lin.weight.grad.detach().clone()
g_theta = thetas.grad.detach().clone()

h = 1e-6
g_fd = torch.zeros_like(g_analytic)
with torch.no_grad():
    for i in range(g_analytic.shape[0]):
        for j in range(g_analytic.shape[1]):
            original = lin.weight[i, j].item()
            lin.weight[i, j] = original + h; loss_plus = float(loss_fn())
            lin.weight[i, j] = original - h; loss_minus = float(loss_fn())
            lin.weight[i, j] = original
            g_fd[i, j] = (loss_plus - loss_minus) / (2 * h)

print(f'max |analytic - central difference| over the classical weights: {(g_analytic - g_fd).abs().max():.2e}')
print(f'smallest |analytic| among the 8 weights:                        {g_analytic.abs().min():.4f}')
print(f'gradient norm on the circuit angles:                            {g_theta.norm():.4f}')


max |analytic - central difference| over the classical weights: 1.22e-10
smallest |analytic| among the 8 weights:                        0.0167
gradient norm on the circuit angles:                            1.9119


## 2) 고전-양자 공동 학습

단일 옵티마이저로 두 파라미터 그룹을 동시에 학습합니다. 두 파라미터 모두 업데이트되며 손실(loss)이 감소합니다.

In [5]:
# 고전 레이어 가중치와 양자 회로 각도를 동시에 학습합니다.


W0 = lin.weight.detach().clone()
theta0 = thetas.detach().clone()
opt = torch.optim.Adam([{'params': lin.parameters()}, {'params': [thetas]}], lr=0.05)

loss_before = float(loss_fn())
for step in range(300):
    opt.zero_grad(set_to_none=True)
    loss = loss_fn()
    loss.backward()
    opt.step()
loss_after = float(loss_fn())

with torch.no_grad():
    prob = torch.clamp((qc.expvals(thetas, embedding=lin(X))[:, 0] + 1) * 0.5, 1e-6, 1 - 1e-6)
    accuracy = ((prob > 0.5).to(dtype) == y).to(dtype).mean().item()

print(f'loss {loss_before:.4f} -> {loss_after:.4f}, training-set accuracy {accuracy * 100:.1f}%')
print(f'|change in classical weights| = {(lin.weight.detach() - W0).norm():.3f}')
print(f'|change in circuit angles|    = {(thetas.detach() - theta0).norm():.3f}')


loss 1.7091 -> 0.2450, training-set accuracy 97.3%
|change in classical weights| = 4.387
|change in circuit angles|    = 0.939


## 3) 앞단 고전 레이어 고정

`embedding.detach()`를 전달하면 고전 레이어가 연산 그래프에서 분리되어 gradient를 받지 않으며, 양자 회로의 각도만 학습됩니다.

In [6]:
# 앞단 고전 레이어의 가중치를 고정하는 표준 방법입니다.


lin.zero_grad(set_to_none=True)
prob = torch.clamp((qc.expvals(thetas, embedding=lin(X).detach())[:, 0] + 1) * 0.5, 1e-6, 1 - 1e-6)
torch.nn.functional.binary_cross_entropy(prob, y).backward()
print('classical weight gradient with embedding.detach():', lin.weight.grad)


classical weight gradient with embedding.detach(): None
